In [1]:
import sys, gc, json, time, pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch

# Make the comparison-search-app package importable so we reuse
# the actual production code (SearchEngine, ParserLLM, VerbalizerLLM).
APP_ROOT = Path(r"C:\Users\shlok\projects\ddp-llm\comparison-search-app")
sys.path.insert(0, str(APP_ROOT))

# Force reload in case an older version is cached
for mod in list(sys.modules):
    if mod.startswith("app."):
        del sys.modules[mod]

from app import config
from app.embeddings import Embeddings
from app.search_engine import SearchEngine
from app.parser_llm import ParserLLM, parsed_to_y
from app.verbalizer_llm import VerbalizerLLM

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB", flush=True)

print(f"config.SIGMA_EPS = {config.SIGMA_EPS}")
print(f"config.MAX_QUERIES = {config.MAX_QUERIES}")
print(f"parser adapter: {config.resolve_parser_adapter()}")

config.SIGMA_EPS = 0.05
config.MAX_QUERIES = 50
parser adapter: C:\Users\shlok\projects\ddp-llm\parser\checkpoints\qwen3b-qlora-v2\checkpoint-final


In [2]:
emb = Embeddings()
print(f"loaded {emb.n} items, dim {emb.X.shape[1]}")
print(f"classes: {emb.classes()}")

# T-06 sampling: every 6th of the 600 → 100 targets
TARGET_INDICES = list(range(0, emb.n, 6))
print(f"sampling {len(TARGET_INDICES)} targets (every 6th)")

# per-target seed so runs are reproducible AND independent
BASE_SEED = 1000
def target_seed(target_idx):
    return BASE_SEED + target_idx

loaded 600 items, dim 10
classes: ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
sampling 100 targets (every 6th)


In [3]:
# Baseline: search consumes oracle answer directly. This is what GAUSSSEARCH
# is designed to run against. Sets the floor for what the closed-loop can do.
# Should reproduce ~15.1 mean queries from T-06.
print("=" * 60)
print("PURE PROBIT BASELINE")
print("=" * 60)

baseline_results = []
t_start = time.time()
for k, target_idx in enumerate(TARGET_INDICES):
    if k % 20 == 0:
        elapsed = time.time() - t_start
        print(f"  {k}/{len(TARGET_INDICES)}  ({elapsed:.1f}s elapsed)", flush=True)
    engine = SearchEngine(
        X=emb.X, sigma_eps=config.SIGMA_EPS,
        target_idx=int(target_idx),
        seed=target_seed(target_idx),
        max_queries=config.MAX_QUERIES,
    )
    while not engine.done:
        i, j = engine.propose_query()
        y = engine.oracle_answer(i, j)
        engine.apply_answer(y, status="clean")
    baseline_results.append({
        "target_idx": target_idx,
        "target_class": emb.label(target_idx),
        "steps": engine.step,
        "stop_reason": engine.stop_reason,
    })

t_baseline = time.time() - t_start
print(f"\ndone in {t_baseline:.1f}s")

steps = np.array([r["steps"] for r in baseline_results])
print(f"\nBaseline results ({len(steps)} targets):")
print(f"  mean:   {steps.mean():.1f}")
print(f"  median: {np.median(steps):.1f}")
print(f"  p95:    {np.percentile(steps, 95):.1f}")
print(f"  max:    {steps.max()}")

# per-class breakdown
per_class = defaultdict(list)
for r in baseline_results:
    per_class[r["target_class"]].append(r["steps"])
print(f"\nper class:")
for cls in sorted(per_class):
    arr = np.array(per_class[cls])
    print(f"  {cls:12s}  n={len(arr)}  mean={arr.mean():.1f}  median={np.median(arr):.1f}")

PURE PROBIT BASELINE
  0/100  (0.0s elapsed)
  20/100  (0.1s elapsed)
  40/100  (0.2s elapsed)
  60/100  (0.3s elapsed)
  80/100  (0.4s elapsed)

done in 0.5s

Baseline results (100 targets):
  mean:   15.1
  median: 15.0
  p95:    25.0
  max:    28

per class:
  buildings     n=17  mean=14.1  median=15.0
  forest        n=17  mean=17.5  median=17.0
  glacier       n=16  mean=15.2  median=13.0
  mountain      n=17  mean=13.6  median=14.0
  sea           n=17  mean=15.4  median=17.0
  street        n=16  mean=14.9  median=16.0


In [7]:
print("=" * 60)
print("CLOSED-LOOP FULL RUN (100 targets, in_query stopping)")
print("=" * 60)

print("loading parser...")
parser = ParserLLM()
free_gpu()
print("loading verbalizer...")
verbalizer = VerbalizerLLM()
free_gpu()

closed_loop_results = []
per_style_stats = defaultdict(lambda: {"match": 0, "skip": 0, "flip": 0, "fell_back": 0})

t_start = time.time()
for k, target_idx in enumerate(TARGET_INDICES):
    if k % 5 == 0:
        elapsed = time.time() - t_start
        eta = elapsed / (k + 1) * (len(TARGET_INDICES) - k - 1) if k > 0 else 0
        print(f"  {k}/{len(TARGET_INDICES)}  ({elapsed:.1f}s elapsed, ETA {eta:.0f}s)", flush=True)

    seed = target_seed(target_idx)
    engine = SearchEngine(
        X=emb.X, sigma_eps=config.SIGMA_EPS,
        target_idx=int(target_idx), seed=seed,
        max_queries=config.MAX_QUERIES,
    )
    rng_lang = np.random.default_rng(seed + 999_999)

    step_log = []
    while not engine.done:
        i, j = engine.propose_query()
        y_oracle = engine.oracle_answer(i, j)
        picked = "A" if y_oracle == 0 else "B"

        utt, style = verbalizer.verbalize(picked, rng_lang, parser=parser)
        parsed, _ = parser.parse(utt, options=("A", "B"))
        y_recovered, y_status = parsed_to_y(parsed)

        base_style = style.replace(" (fallback)", "")
        fell_back = "(fallback)" in style
        if fell_back:
            per_style_stats[base_style]["fell_back"] += 1

        if y_recovered is None:
            status = "skip"
            per_style_stats[base_style]["skip"] += 1
        elif y_recovered == y_oracle:
            status = "match"
            per_style_stats[base_style]["match"] += 1
        else:
            status = "flip"
            per_style_stats[base_style]["flip"] += 1

        if status == "skip":
            engine.apply_answer(None, status="skip")
        else:
            engine.apply_answer(y_recovered, status=status)

        step_log.append({
            "step": engine.step, "picked": picked, "utt": utt,
            "style": style, "status": status, "y_oracle": y_oracle,
            "y_recovered": y_recovered,
        })

    closed_loop_results.append({
        "target_idx": target_idx,
        "target_class": emb.label(target_idx),
        "steps": engine.step,
        "stop_reason": engine.stop_reason,
        "step_log": step_log,
    })

t_closed = time.time() - t_start
print(f"\ndone in {t_closed:.1f}s ({t_closed/60:.1f} min)")

with open(Path(r"C:\Users\shlok\projects\ddp-llm\scenery-search\notebooks") /
           "closed_loop_100_v5_gated_fixed.pkl", "wb") as f:
    pickle.dump({
        "baseline_results": baseline_results,
        "closed_loop_results": closed_loop_results,
        "per_style_stats": dict(per_style_stats),
    }, f)
print("saved to closed_loop_100_v5_gated_fixed.pkl")

del verbalizer, parser
free_gpu()

CLOSED-LOOP FULL RUN (100 targets, in_query stopping)
loading parser...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

VRAM: 2.19 GB
loading verbalizer...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

VRAM: 5.28 GB
  0/100  (0.0s elapsed, ETA 0s)
  5/100  (128.6s elapsed, ETA 2015s)
  10/100  (234.7s elapsed, ETA 1899s)
  15/100  (329.1s elapsed, ETA 1728s)
  20/100  (431.1s elapsed, ETA 1622s)
  25/100  (593.0s elapsed, ETA 1688s)
  30/100  (711.0s elapsed, ETA 1582s)
  35/100  (842.9s elapsed, ETA 1498s)
  40/100  (967.7s elapsed, ETA 1393s)
  45/100  (1043.1s elapsed, ETA 1225s)
  50/100  (1168.8s elapsed, ETA 1123s)
  55/100  (1262.1s elapsed, ETA 992s)
  60/100  (1378.6s elapsed, ETA 881s)
  65/100  (1480.2s elapsed, ETA 763s)
  70/100  (1594.3s elapsed, ETA 651s)
  75/100  (1732.9s elapsed, ETA 547s)
  80/100  (1807.2s elapsed, ETA 424s)
  85/100  (1942.2s elapsed, ETA 316s)
  90/100  (2062.2s elapsed, ETA 204s)
  95/100  (2172.8s elapsed, ETA 91s)

done in 2269.5s (37.8 min)
saved to closed_loop_100_v5_gated_fixed.pkl
VRAM: 0.01 GB


In [6]:
# Sanity run: just 5 targets to verify the fix before the full 100-target run.
print("=" * 60)
print("SANITY CHECK — 5 targets")
print("=" * 60)

print("loading parser...")
parser = ParserLLM()
free_gpu()
print("loading verbalizer...")
verbalizer = VerbalizerLLM()
free_gpu()

sanity_targets = TARGET_INDICES[:5]
sanity_results = []
t_start = time.time()
for k, target_idx in enumerate(sanity_targets):
    seed = target_seed(target_idx)
    engine = SearchEngine(
        X=emb.X, sigma_eps=config.SIGMA_EPS,
        target_idx=int(target_idx), seed=seed,
        max_queries=config.MAX_QUERIES,
    )
    rng_lang = np.random.default_rng(seed + 999_999)
    step_log = []
    while not engine.done:
        i, j = engine.propose_query()
        y_oracle = engine.oracle_answer(i, j)
        picked = "A" if y_oracle == 0 else "B"
        utt, style = verbalizer.verbalize(picked, rng_lang, parser=parser)
        parsed, _ = parser.parse(utt, options=("A", "B"))
        y_recovered, y_status = parsed_to_y(parsed)

        if y_recovered is None:
            status = "skip"
        elif y_recovered == y_oracle:
            status = "match"
        else:
            status = "flip"

        step_log.append({
            "picked": picked, "utt": utt, "style": style,
            "y_oracle": y_oracle, "y_recovered": y_recovered, "status": status,
        })
        if status == "skip":
            engine.apply_answer(None, status="skip")
        else:
            engine.apply_answer(y_recovered, status=status)

    sanity_results.append({"target_idx": target_idx, "steps": engine.step, "step_log": step_log})
    print(f"  target {target_idx}: {engine.step} steps, "
          f"match={sum(1 for s in step_log if s['status']=='match')}, "
          f"flip={sum(1 for s in step_log if s['status']=='flip')}, "
          f"skip={sum(1 for s in step_log if s['status']=='skip')}")

elapsed = time.time() - t_start
print(f"\ndone in {elapsed:.1f}s")

# Show a few utterances from target 0 for eyeballing
print(f"\nsample steps from first target (idx={sanity_results[0]['target_idx']}):")
for s in sanity_results[0]["step_log"][:8]:
    print(f"  picked={s['picked']}  style={s['style']:12s}  utt={s['utt']!r:40s}  y_oracle={s['y_oracle']} y_recovered={s['y_recovered']} status={s['status']}")

del verbalizer, parser
free_gpu()

SANITY CHECK — 5 targets
loading parser...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

VRAM: 2.19 GB
loading verbalizer...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

VRAM: 5.28 GB
  target 0: 12 steps, match=12, flip=0, skip=0
  target 6: 13 steps, match=13, flip=0, skip=0
  target 12: 15 steps, match=15, flip=0, skip=0
  target 18: 20 steps, match=20, flip=0, skip=0
  target 24: 19 steps, match=19, flip=0, skip=0

done in 116.7s

sample steps from first target (idx=0):
  picked=A  style=direct        utt='A for sure.'                             y_oracle=0 y_recovered=0 status=match
  picked=A  style=direct        utt='A.'                                      y_oracle=0 y_recovered=0 status=match
  picked=B  style=positional    utt='the right one'                           y_oracle=1 y_recovered=1 status=match
  picked=A  style=casual        utt='Got it! First one.'                      y_oracle=0 y_recovered=0 status=match
  picked=B  style=positional    utt='the right one'                           y_oracle=1 y_recovered=1 status=match
  picked=B  style=direct        utt='B it is.'                                y_oracle=1 y_recovered=1 status=m

In [8]:
# Summary stats + comparison against T-06 baseline (from session 5).
print("=" * 70)
print("CLOSED-LOOP RESULTS (100 targets, in_query stopping)")
print("=" * 70)

steps_baseline = np.array([r["steps"] for r in baseline_results])
steps_closed = np.array([r["steps"] for r in closed_loop_results])

print(f"\n{'metric':<15s} {'baseline':<12s} {'closed-loop':<14s} {'delta':<10s}")
print("-" * 55)
for name, fn in [("mean", np.mean), ("median", np.median),
                  ("p95", lambda x: np.percentile(x, 95)),
                  ("max", np.max)]:
    b = fn(steps_baseline)
    c = fn(steps_closed)
    print(f"{name:<15s} {b:<12.1f} {c:<14.1f} {c-b:+.1f}")

# Round-trip fidelity
total_steps = sum(len(r["step_log"]) for r in closed_loop_results)
n_match = sum(1 for r in closed_loop_results for s in r["step_log"] if s["status"] == "match")
n_flip = sum(1 for r in closed_loop_results for s in r["step_log"] if s["status"] == "flip")
n_skip = sum(1 for r in closed_loop_results for s in r["step_log"] if s["status"] == "skip")

print(f"\nround-trip fidelity across all {total_steps} steps:")
print(f"  match:  {n_match:4d}  ({100*n_match/total_steps:.1f}%)")
print(f"  flip:   {n_flip:4d}  ({100*n_flip/total_steps:.1f}%)")
print(f"  skip:   {n_skip:4d}  ({100*n_skip/total_steps:.1f}%)")

# Per-style stats
print(f"\nper-style breakdown:")
print(f"  {'style':<12s} {'match':<8s} {'flip':<8s} {'skip':<8s} {'fell_back':<10s} {'match%':<8s}")
print("-" * 60)
for style_name in ["letter", "ordinal", "positional", "direct", "casual"]:
    if style_name not in per_style_stats:
        continue
    s = per_style_stats[style_name]
    n = s["match"] + s["flip"] + s["skip"]
    match_pct = 100 * s["match"] / n if n > 0 else 0
    print(f"  {style_name:<12s} {s['match']:<8d} {s['flip']:<8d} {s['skip']:<8d} {s['fell_back']:<10d} {match_pct:.1f}%")

# Per-class
per_class_b = defaultdict(list)
per_class_c = defaultdict(list)
for r in baseline_results:
    per_class_b[r["target_class"]].append(r["steps"])
for r in closed_loop_results:
    per_class_c[r["target_class"]].append(r["steps"])

print(f"\nper-class comparison:")
print(f"  {'class':<12s} {'baseline':<12s} {'closed':<12s} {'delta':<10s}")
print("-" * 50)
for cls in sorted(per_class_b):
    b = np.mean(per_class_b[cls])
    c = np.mean(per_class_c[cls])
    print(f"  {cls:<12s} {b:<12.1f} {c:<12.1f} {c-b:+.1f}")

# vs old T-06 numbers (from session 5 handover)
print(f"\n\n=== NEW STACK vs OLD T-06 RESULTS ===")
print(f"                  {'T-06 old':<12s} {'new':<12s} {'delta':<10s}")
print(f"baseline mean:    {'15.1':<12s} {steps_baseline.mean():<12.1f} {'(sanity)':<10s}")
print(f"closed-loop mean: {'19.2':<12s} {steps_closed.mean():<12.1f} {steps_closed.mean() - 19.2:+.1f}")
print(f"round-trip match: {'87.7%':<12s} {100*n_match/total_steps:<11.1f}% {100*n_match/total_steps - 87.7:+.1f}pp")
print(f"round-trip flip:  {'~10%':<12s} {100*n_flip/total_steps:<11.1f}%")
print(f"round-trip skip:  {'~2%':<12s} {100*n_skip/total_steps:<11.1f}%")

CLOSED-LOOP RESULTS (100 targets, in_query stopping)

metric          baseline     closed-loop    delta     
-------------------------------------------------------
mean            15.1         15.1           +0.0
median          15.0         15.0           +0.0
p95             25.0         25.0           +0.0
max             28.0         28.0           +0.0

round-trip fidelity across all 1510 steps:
  match:  1510  (100.0%)
  flip:      0  (0.0%)
  skip:      0  (0.0%)

per-style breakdown:
  style        match    flip     skip     fell_back  match%  
------------------------------------------------------------
  letter       296      0        0        9          100.0%
  ordinal      321      0        0        0          100.0%
  positional   282      0        0        0          100.0%
  direct       294      0        0        0          100.0%
  casual       317      0        0        0          100.0%

per-class comparison:
  class        baseline     closed       delta     
----

In [9]:
# Compare first target's (i, j) trajectory in baseline vs closed loop.
# If truly identical: verbalizer/parser are a clean pipe. If not: something
# in the closed loop is drifting the search engine's RNG.

target_idx = TARGET_INDICES[0]
seed = target_seed(target_idx)

# Baseline trajectory
engine_b = SearchEngine(X=emb.X, sigma_eps=config.SIGMA_EPS,
                        target_idx=int(target_idx), seed=seed,
                        max_queries=config.MAX_QUERIES)
baseline_trajectory = []
while not engine_b.done:
    i, j = engine_b.propose_query()
    y = engine_b.oracle_answer(i, j)
    baseline_trajectory.append((engine_b.step, i, j, y))
    engine_b.apply_answer(y, status="clean")

# Closed-loop trajectory from the saved run
closed_trajectory = [(s["step"] - 1, None, None, s["y_oracle"])
                     for s in closed_loop_results[0]["step_log"]]
# We don't have (i, j) saved from closed loop, so compare y-sequences
print(f"target_idx={target_idx}")
print(f"baseline y-sequence:    {[t[3] for t in baseline_trajectory]}")
print(f"closed-loop y-sequence: {[t[3] for t in closed_trajectory]}")
print(f"match: {[t[3] for t in baseline_trajectory] == [t[3] for t in closed_trajectory]}")

# Show utterances and picked letters side-by-side
print(f"\nclosed-loop utterances:")
for s in closed_loop_results[0]["step_log"]:
    print(f"  step={s['step']}  picked={s['picked']}  y_oracle={s['y_oracle']}  utt={s['utt']!r}  status={s['status']}")

target_idx=0
baseline y-sequence:    [0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0]
closed-loop y-sequence: [0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0]
match: True

closed-loop utterances:
  step=1  picked=A  y_oracle=0  utt='A for sure.'  status=match
  step=2  picked=A  y_oracle=0  utt='A.'  status=match
  step=3  picked=B  y_oracle=1  utt='the right one'  status=match
  step=4  picked=A  y_oracle=0  utt='Got it! First one.'  status=match
  step=5  picked=B  y_oracle=1  utt='the right one'  status=match
  step=6  picked=B  y_oracle=1  utt='B it is.'  status=match
  step=7  picked=B  y_oracle=1  utt='the second one'  status=match
  step=8  picked=A  y_oracle=0  utt='A'  status=match
  step=9  picked=B  y_oracle=1  utt='the second one'  status=match
  step=10  picked=A  y_oracle=0  utt='I’ll go with A.'  status=match
  step=11  picked=B  y_oracle=1  utt='the second one'  status=match
  step=12  picked=A  y_oracle=0  utt='Right choice, A!'  status=match


In [10]:
# σ_ε (Probit noise) flip rate: how often does the noisy oracle disagree with
# the noiseless geometric answer.
# The noiseless answer is which side of the bisecting hyperplane the target
# actually sits on (probit_prob=1 exactly). σ_ε=0.05 lets the answer flip when
# the target is close to that hyperplane.

from app.search_engine import bisecting_hyperplane

print("=" * 60)
print("σ_ε NOISE — noisy oracle vs noiseless geometry")
print("=" * 60)

n_agree = 0
n_disagree = 0
per_target_disagree = defaultdict(int)

for r in closed_loop_results:
    target_idx = r["target_idx"]
    x_t = emb.X[target_idx]

    # We need to reconstruct (i, j) at each step. Do it by re-running the
    # engine with the same seed (deterministic modulo the oracle noise being
    # consumed identically, which it was since y_oracle matches).
    seed = target_seed(target_idx)
    engine = SearchEngine(X=emb.X, sigma_eps=config.SIGMA_EPS,
                          target_idx=int(target_idx), seed=seed,
                          max_queries=config.MAX_QUERIES)
    for step_log in r["step_log"]:
        i, j = engine.propose_query()
        x_i, x_j = emb.X[i], emb.X[j]
        w, b = bisecting_hyperplane(x_i, x_j)
        # noiseless answer: 0 if target is closer to i (i.e. on i's side)
        signed_dist = np.dot(x_t, w) + b
        y_noiseless = 0 if signed_dist > 0 else 1
        y_noisy = step_log["y_oracle"]

        if y_noiseless == y_noisy:
            n_agree += 1
        else:
            n_disagree += 1
            per_target_disagree[target_idx] += 1

        engine.apply_answer(y_noisy, status="clean")

total = n_agree + n_disagree
print(f"\ntotal steps across 100 targets: {total}")
print(f"  agree with noiseless:    {n_agree}  ({100*n_agree/total:.1f}%)")
print(f"  disagree (σ_ε flip):     {n_disagree}  ({100*n_disagree/total:.1f}%)")

# distribution of flips per target
flip_counts = [per_target_disagree.get(r["target_idx"], 0) for r in closed_loop_results]
print(f"\nper-target flip distribution:")
print(f"  mean flips per target:   {np.mean(flip_counts):.2f}")
print(f"  targets with 0 flips:    {sum(1 for f in flip_counts if f == 0)}/100")
print(f"  targets with 1 flip:     {sum(1 for f in flip_counts if f == 1)}/100")
print(f"  targets with 2+ flips:   {sum(1 for f in flip_counts if f >= 2)}/100")
print(f"  max flips on one target: {max(flip_counts)}")

print(f"\n\n=== FULL PICTURE: noise decomposition ===")
print(f"σ_ε Probit oracle:     {100*n_disagree/total:.1f}% flip rate  (intentional, algorithm-designed)")
print(f"Verbalizer→parser:      0.0% flip rate  (validation gate makes it lossless)")
print(f"Total closed-loop flip: {100*n_disagree/total:.1f}%  (all from σ_ε)")

σ_ε NOISE — noisy oracle vs noiseless geometry


RuntimeError: search already finished

In [11]:
lens = [len(r["step_log"]) for r in closed_loop_results]
print(f"step_log lengths: min={min(lens)}, max={max(lens)}, mean={np.mean(lens):.1f}")
print(f"n targets: {len(closed_loop_results)}")
print(f"first target step_log length: {len(closed_loop_results[0]['step_log'])}")
print(f"total steps: {sum(lens)}")

step_log lengths: min=3, max=28, mean=15.1
n targets: 100
first target step_log length: 12
total steps: 1510


In [12]:
from app.search_engine import bisecting_hyperplane

print("=" * 60)
print("σ_ε NOISE — noisy oracle vs noiseless geometry")
print("=" * 60)

n_agree = 0
n_disagree = 0
per_target_disagree = defaultdict(int)
skipped_targets = []

for r in closed_loop_results:
    target_idx = r["target_idx"]
    x_t = emb.X[target_idx]

    seed = target_seed(target_idx)
    engine = SearchEngine(X=emb.X, sigma_eps=config.SIGMA_EPS,
                          target_idx=int(target_idx), seed=seed,
                          max_queries=config.MAX_QUERIES)

    for step_log in r["step_log"]:
        if engine.done:
            # Reconstruction diverged from the recorded run — skip remaining
            # steps for this target. Rare, doesn't materially affect aggregate.
            skipped_targets.append((target_idx, len(r["step_log"]) - engine.step))
            break

        i, j = engine.propose_query()
        x_i, x_j = emb.X[i], emb.X[j]
        w, b = bisecting_hyperplane(x_i, x_j)
        signed_dist = np.dot(x_t, w) + b
        y_noiseless = 0 if signed_dist > 0 else 1
        y_noisy = step_log["y_oracle"]

        if y_noiseless == y_noisy:
            n_agree += 1
        else:
            n_disagree += 1
            per_target_disagree[target_idx] += 1

        engine.apply_answer(y_noisy, status="clean")

total = n_agree + n_disagree
print(f"\ntotal steps analyzed: {total}")
print(f"  agree with noiseless:    {n_agree}  ({100*n_agree/total:.1f}%)")
print(f"  disagree (σ_ε flip):     {n_disagree}  ({100*n_disagree/total:.1f}%)")

if skipped_targets:
    print(f"\nWARNING: reconstruction diverged for {len(skipped_targets)} targets")
    print(f"  (skipped some steps — aggregate is approximate)")
    for tid, n_skipped in skipped_targets[:5]:
        print(f"    target {tid}: skipped {n_skipped} steps")

flip_counts = [per_target_disagree.get(r["target_idx"], 0) for r in closed_loop_results]
print(f"\nper-target flip distribution:")
print(f"  mean flips per target:   {np.mean(flip_counts):.2f}")
print(f"  targets with 0 flips:    {sum(1 for f in flip_counts if f == 0)}/100")
print(f"  targets with 1 flip:     {sum(1 for f in flip_counts if f == 1)}/100")
print(f"  targets with 2+ flips:   {sum(1 for f in flip_counts if f >= 2)}/100")
print(f"  max flips on one target: {max(flip_counts)}")

print(f"\n=== FULL PICTURE: noise decomposition ===")
print(f"σ_ε Probit oracle:     {100*n_disagree/total:.1f}% flip rate  (intentional, algorithm-designed)")
print(f"Verbalizer→parser:      0.0% flip rate  (validation gate makes it lossless)")
print(f"Total closed-loop flip: {100*n_disagree/total:.1f}%  (all from σ_ε)")

σ_ε NOISE — noisy oracle vs noiseless geometry

total steps analyzed: 1460
  agree with noiseless:    755  (51.7%)
  disagree (σ_ε flip):     705  (48.3%)

  (skipped some steps — aggregate is approximate)
    target 174: skipped 15 steps
    target 222: skipped 4 steps
    target 246: skipped 10 steps
    target 288: skipped 10 steps
    target 324: skipped 4 steps

per-target flip distribution:
  mean flips per target:   7.05
  targets with 0 flips:    1/100
  targets with 1 flip:     4/100
  targets with 2+ flips:   95/100
  max flips on one target: 17

=== FULL PICTURE: noise decomposition ===
σ_ε Probit oracle:     48.3% flip rate  (intentional, algorithm-designed)
Verbalizer→parser:      0.0% flip rate  (validation gate makes it lossless)
Total closed-loop flip: 48.3%  (all from σ_ε)


In [13]:
from app.search_engine import bisecting_hyperplane

print("=" * 60)
print("σ_ε NOISE — noisy oracle vs noiseless geometry")
print("=" * 60)

n_agree = 0
n_disagree = 0
per_target_disagree = defaultdict(int)

for r in closed_loop_results:
    target_idx = r["target_idx"]
    x_t = emb.X[target_idx]

    seed = target_seed(target_idx)
    engine = SearchEngine(X=emb.X, sigma_eps=config.SIGMA_EPS,
                          target_idx=int(target_idx), seed=seed,
                          max_queries=config.MAX_QUERIES)

    while not engine.done:
        i, j = engine.propose_query()
        x_i, x_j = emb.X[i], emb.X[j]
        w, b = bisecting_hyperplane(x_i, x_j)
        signed_dist = np.dot(x_t, w) + b
        y_noiseless = 0 if signed_dist > 0 else 1

        # Call oracle_answer to consume RNG the same way the real run did.
        y_noisy = engine.oracle_answer(i, j)

        if y_noiseless == y_noisy:
            n_agree += 1
        else:
            n_disagree += 1
            per_target_disagree[target_idx] += 1

        engine.apply_answer(y_noisy, status="clean")

total = n_agree + n_disagree
print(f"\ntotal steps: {total}")
print(f"  agree with noiseless:    {n_agree}  ({100*n_agree/total:.1f}%)")
print(f"  disagree (σ_ε flip):     {n_disagree}  ({100*n_disagree/total:.1f}%)")

flip_counts = [per_target_disagree.get(r["target_idx"], 0) for r in closed_loop_results]
print(f"\nper-target flip distribution:")
print(f"  mean flips per target:   {np.mean(flip_counts):.2f}")
print(f"  targets with 0 flips:    {sum(1 for f in flip_counts if f == 0)}/100")
print(f"  targets with 1 flip:     {sum(1 for f in flip_counts if f == 1)}/100")
print(f"  targets with 2+ flips:   {sum(1 for f in flip_counts if f >= 2)}/100")
print(f"  max flips on one target: {max(flip_counts)}")

# Also compute how confident the oracle is on average (helps interpret)
# by looking at probit_prob at each query.
from app.search_engine import probit_prob

confidences = []
for r in closed_loop_results:
    target_idx = r["target_idx"]
    x_t = emb.X[target_idx]
    seed = target_seed(target_idx)
    engine = SearchEngine(X=emb.X, sigma_eps=config.SIGMA_EPS,
                          target_idx=int(target_idx), seed=seed,
                          max_queries=config.MAX_QUERIES)
    while not engine.done:
        i, j = engine.propose_query()
        p = probit_prob(emb.X[i], emb.X[j], x_t, config.SIGMA_EPS)
        confidences.append(max(p, 1 - p))  # confidence in the more likely answer
        y = engine.oracle_answer(i, j)
        engine.apply_answer(y, status="clean")

confidences = np.array(confidences)
print(f"\noracle confidence per query:")
print(f"  mean:              {confidences.mean():.3f}")
print(f"  median:            {np.median(confidences):.3f}")
print(f"  fraction >0.9:     {(confidences > 0.9).mean():.2f}")
print(f"  fraction <0.6:     {(confidences < 0.6).mean():.2f}  (near-coinflip queries)")

print(f"\n=== NOISE DECOMPOSITION ===")
print(f"σ_ε Probit oracle:     {100*n_disagree/total:.1f}% flip vs noiseless geometry")
print(f"Verbalizer→parser:      0.0% flip  (validation gate is lossless)")

σ_ε NOISE — noisy oracle vs noiseless geometry

total steps: 1510
  agree with noiseless:    1494  (98.9%)
  disagree (σ_ε flip):     16  (1.1%)

per-target flip distribution:
  mean flips per target:   0.16
  targets with 0 flips:    86/100
  targets with 1 flip:     12/100
  targets with 2+ flips:   2/100
  max flips on one target: 2

oracle confidence per query:
  mean:              0.987
  median:            1.000
  fraction >0.9:     0.96
  fraction <0.6:     0.01  (near-coinflip queries)

=== NOISE DECOMPOSITION ===
σ_ε Probit oracle:     1.1% flip vs noiseless geometry
Verbalizer→parser:      0.0% flip  (validation gate is lossless)
